# ResumeMatch: the RAG pipeline, step by step

This notebook runs the **real code of the app** (`backend/rag/`) one step at a time, so you can see what every stage produces.

**How to run:** click **Run All** at the top, or run each cell with **Shift + Enter**.

| Step | What it does | File |
|---|---|---|
| 1 | Load the PDF and repair its text | `rag/loaders.py` |
| 2 | Section-aware chunking | `rag/chunker.py` |
| 3 | Embeddings (text → vectors) | `rag/embedder.py` |
| 4 | Keyword search (BM25) | `rag/index.py` |
| 5 | Semantic search (cosine similarity) | `rag/index.py` |
| 6 | Hybrid search with Reciprocal Rank Fusion | `rag/index.py` |
| 7 | Cross-encoder reranking | `rag/reranker.py` |
| 8 | Requirement extraction (LLM) | `rag/analyzer.py` |
| 9 | HyDE query expansion (LLM) | `rag/analyzer.py` |
| 10 | Judgement (LLM) + hallucination check | `rag/analyzer.py`, `rag/grounding.py` |
| 11 | Score | `rag/analyzer.py` |
| 12 | Evaluation / ablation study | `rag/evaluation.py` |

## 0. Setup: connect the notebook to the app's code

In [ ]:
import os, sys
from pathlib import Path

BACKEND = Path.cwd().parent / "backend"          # notebooks/ is next to backend/
sys.path.insert(0, str(BACKEND))
os.chdir(BACKEND)                                # so .env, sample_docs/ and model_cache/ are found
os.environ.setdefault("DJANGO_SETTINGS_MODULE", "config.settings")

import django
django.setup()
from django.conf import settings

print("Backend folder :", BACKEND)
print("Gemini key set :", bool(settings.GEMINI_API_KEY))
print("Gemini model   :", settings.GEMINI_MODEL)

# small helper to show results as a table (no pandas needed)
from html import escape
from IPython.display import HTML, display
def table(rows):
    if not rows:
        return print("(no rows)")
    cols = list(rows[0].keys())
    head = "".join(f"<th style='text-align:left'>{escape(str(c))}</th>" for c in cols)
    body = "".join("<tr>" + "".join(f"<td style='text-align:left'>{escape(str(r.get(c, '')))}</td>" for c in cols) + "</tr>" for r in rows)
    display(HTML(f"<table><tr>{head}</tr>{body}</table>"))

### Choose the documents
By default we use the sample resume and job description. To use **your own CV**, put its full path in `RESUME_FILE`, e.g.
`RESUME_FILE = r"C:\Users\you\Downloads\SAAKSHI_LOKHANDE_CV.pdf"`

In [ ]:
RESUME_FILE = settings.SAMPLE_DIR / "sample_resume.txt"
JD_FILE     = settings.SAMPLE_DIR / "sample_job_description.txt"

resume_bytes = Path(RESUME_FILE).read_bytes()
jd_bytes     = Path(JD_FILE).read_bytes()
print(Path(RESUME_FILE).name, "|", Path(JD_FILE).name)

## Step 1: Load the documents and repair the text
PDFs store text as positioned letters. Extraction sometimes adds wrong spaces (`F AISS`, `T ools`) because of letter-spacing, so `repair_pdf_spacing()` fixes that.

In [ ]:
from rag.loaders import extract_pages, repair_pdf_spacing

print("Repair demo:", repair_pdf_spacing("T ools: Git, F AISS, F acial analysis | A strong team"))

resume_pages = extract_pages(Path(RESUME_FILE).name, resume_bytes)
jd_pages     = extract_pages(Path(JD_FILE).name, jd_bytes)
resume_text  = "\n\n".join(t for _, t in resume_pages)
jd_text      = "\n\n".join(t for _, t in jd_pages)

print(f"\nResume: {len(resume_pages)} page(s), {len(resume_text)} characters")
print(resume_text[:600])

## Step 2: Section-aware chunking
We detect headings (Experience, Projects, Skills…), then split each section into ~500-character chunks with 100 characters of **overlap**, so an idea cut at a boundary still appears whole in one chunk. Chunks never mix two sections.

In [ ]:
from rag.chunker import chunk_pages, detect_section

for line in ["Technical Skills", "EXPERIENCE", "Preferred qualifications (nice to have)", "Built a Django REST API"]:
    print(f"{line!r:45} -> heading: {detect_section(line)}")

chunks = (chunk_pages(resume_pages, "resume", settings.CHUNK_SIZE, settings.CHUNK_OVERLAP)
          + chunk_pages(jd_pages, "jd", settings.CHUNK_SIZE, settings.CHUNK_OVERLAP))

table([{"id": c["id"], "section": c["section"], "chars": len(c["text"]),
               "text": c["text"].replace("\n", " / ")} for c in chunks])

## Step 3: Embeddings (text → numbers that capture meaning)
Model: **BAAI/bge-small-en-v1.5**, which turns each chunk into **384 numbers**. Texts with similar meaning get similar vectors. We normalise the vectors, so **cosine similarity = dot product**.

*The first run downloads the model (~130 MB), which takes about a minute.*

In [ ]:
from rag import embedder
import numpy as np

vectors = embedder.embed_passages([f"{c['section']}: {c['text']}" for c in chunks])
print("Vectors shape:", vectors.shape, "(chunks x dimensions)")

# Meaning vs words: these phrases share almost no words, but their meanings are related
a, b, c = embedder.embed_passages(["machine learning engineer", "built neural network models", "baked a chocolate cake"])
print(f"'ML engineer' vs 'neural network models': {a @ b:.3f}")
print(f"'ML engineer' vs 'chocolate cake'       : {a @ c:.3f}")

## Step 4: Keyword search with BM25
BM25 scores a chunk higher when it contains the query words, especially **rare** words (via IDF), and does not over-reward repetition. It's great for exact skills like `FAISS`, `SQL` or `Kinesis`.

In [ ]:
from rag.index import HybridIndex, tokenize

index = HybridIndex(chunks, vectors)
query = "experience with containers and deployment"
print("Query tokens:", tokenize(query))

ids = index._candidates("resume")
ranked, scores = index.keyword_rank(query, ids)
for i in ranked[:3]:
    print(f"\nBM25 {scores[i]:.2f} | {chunks[i]['section']} | {chunks[i]['text'][:120]}")

## Step 5: Semantic search (cosine similarity between vectors)

In [ ]:
ranked_sem, sims = index.semantic_rank(query, ids)
for i in ranked_sem[:3]:
    print(f"\ncosine {sims[i]:.3f} | {chunks[i]['section']} | {chunks[i]['text'][:120]}")

## Step 6: Hybrid search with Reciprocal Rank Fusion (RRF)
BM25 scores and cosine scores are on different scales, so we combine **ranks** instead of scores:

$$ RRF(d) = \sum_{	ext{lists}} rac{1}{60 + 	ext{rank}(d)} $$

A chunk ranked well by *both* methods rises to the top.

In [ ]:
results = index.search(query, k=5, source="resume", mode="hybrid")
table([{"rrf": r["score"], "bm25": r["keyword"], "cosine": r["semantic"],
               "section": r["section"], "text": r["text"][:80].replace("\n", " ")} for r in results])

## Step 7: Cross-encoder reranking (stage 2)
Stage 1 encodes query and chunk **separately**, which is fast but approximate. A **cross-encoder** (`ms-marco-MiniLM-L-6-v2`) reads them **together**, which is more precise but slower, so we only use it on the top 15 candidates.

In [ ]:
from rag import reranker

reranked = index.search(query, k=5, source="resume", mode="hybrid", rerank=True)
table([{"rerank_relevance": r["rerank"], "rrf": r["score"], "section": r["section"],
               "text": r["text"][:80].replace("\n", " ")} for r in reranked])

## Step 8: Requirement extraction (LLM)
From here on we need the **Gemini API key** (in `backend/.env`). The LLM reads the job description and returns structured JSON.

In [ ]:
from rag import analyzer, llm
assert llm.configured(), "Add GEMINI_API_KEY to backend/.env and restart the kernel"

role, reqs = analyzer.extract_requirements(jd_text)
print("Role:", role)
table(reqs)

## Step 9: HyDE (Hypothetical Document Embeddings)
Job ads and resumes use different words ("containerization expertise" vs "Dockerized the app"). HyDE asks the LLM to **imagine a resume line** that would satisfy each requirement, and we search with it **as well as** the requirement.

In [ ]:
hyde = analyzer.hyde_queries(reqs)
for i, r in enumerate(reqs[:5]):
    print(f"REQUIREMENT : {r['requirement']}\nHyDE LINE   : {hyde.get(i)}\n")

## Step 10: Retrieve evidence → LLM judgement → hallucination check

In [ ]:
analyzer.retrieve_evidence(index, reqs, hyde=hyde, rerank=True)
print("Evidence retrieved for requirement 1:")
for c in reqs[0]["evidence_chunks"]:
    print(f"  [{c['section']}] rerank={c['rerank']} | {c['text'][:100]}")

judged = analyzer.judge_requirements(reqs)
analyzer.apply_judgement(reqs, judged, index, resume_text)

table([{"requirement": r["requirement"], "type": r["type"], "status": r["status"],
               "verified": r["verified"], "similarity": r["similarity"], "evidence": r["evidence"]} for r in reqs])

### How the hallucination check works
Every quote is fuzzy-matched against the resume. Similarity ≥ 0.80 means **verified**. Below that, the claim is flagged, and a "match" is downgraded to "partial".

In [ ]:
from rag import grounding
tests = [
    "Deployed the model as a Django REST API",            # real (sample resume)
    "deployed the model as a django rest api service",    # small paraphrase
    "Led a team of 10 engineers building Kubernetes clusters",  # invented
]
for q in tests:
    print(grounding.verify(q, resume_text), "|", q)

## Step 11: The score (computed in Python, not by the LLM)
must-have = weight 2, nice-to-have = weight 1; match = 1, partial = 0.5, missing = 0

In [ ]:
print("Match score     :", analyzer.compute_score(reqs), "%")
print("Grounding rate  :", analyzer.grounding_rate(reqs), "% of quotes verified")
print("Semantic sim.   :", analyzer.semantic_similarity(index), "%")
print("\nSummary:", judged.get("summary"))

## Step 12: Evaluation (ablation study)
Hand-labelled gold data. We add one component at a time and measure **Hit@1, Hit@3, MRR, nDCG@5**, then the LLM's **accuracy / macro-F1** against gold verdicts.

In [ ]:
from rag import evaluation
res = evaluation.run()
for n in res["notes"]:
    print("note:", n)
print("A. Requirement -> evidence retrieval")
table(res["requirements"]["rows"])
print("B. Chat question retrieval")
table(res["qa"]["rows"])
if res["judgement"]:
    j = res["judgement"]
    print(f"C. Judgement: accuracy {j['accuracy']}% | macro-F1 {j['macro_f1']} | grounding {j['grounding_rate']}%")
    table([{"gold \\ predicted": g, **j["matrix"][g]} for g in j["labels"]])

## Summary
**Load → Chunk (section-aware) → Embed → Hybrid search (BM25 + vectors, RRF) + HyDE → Cross-encoder rerank → LLM judge → Verify → Score → Evaluate**

The web app runs exactly this code: `POST /api/analyze/` calls `rag/pipeline.py` → `rag/analyzer.py`.